### **Vision-Language Medical Foundation Models**

#### **1.5. Few-shot Parameter Efficient Fine-Tuning**
---

Parameter-Efficient Fine-Tuning is a methodology of increasing interest, currently popularized in the NLP community to adapt the recently introduced large-scale LLMs.

**Objective**: Given an small set of examples per category, we want to fine-tune parts of the model to specialize it on a particular task. By tuning only few parameters, we can adapt large-complexity models with minimal resources.

**Few-shot**: We only use K number of images for each new category.

**Why PEFT?**: They are efficient (ar least more than full fine-tuning), usually run with in-house GPUs. They are fast: you can transfer the model in a matter of minutes. They are more flexible than black-box Adapters, since it allows to refine deep features.

In [1]:
# If a package import fails, uncomment/run this line once, then restart the kernel.
# %pip install -q openpyxl transformers pandas numpy matplotlib scikit-learn tqdm pillow

# General imports.
import warnings
warnings.filterwarnings('ignore')

import copy
import shutil
from pathlib import Path

import torch
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from PIL import Image

# Device for training/inference.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Available device: " + device)

# Seeds for reproducibility.
def set_seeds(seed_value, use_cuda):
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    random.seed(seed_value)
    if use_cuda:
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seeds(42, use_cuda=torch.cuda.is_available())

Available device: cuda


#### **Dataset details**

In [2]:
# SICAPv2 dataset metadata.
# One copy is stored in the shared project folder. Each student copies it once to local_data.
SHARED_DATA_ROOT = Path.home() / "projects" / "def-sponsor00" / "SICAPv2"
LOCAL_DATA_ROOT = Path.home() / "local_data" / "datasets" / "SICAPv2"

if not LOCAL_DATA_ROOT.exists():
    LOCAL_DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    print("Copying SICAPv2 from shared project folder to local_data...")
    shutil.copytree(SHARED_DATA_ROOT, LOCAL_DATA_ROOT)
else:
    print("SICAPv2 already exists in local_data.")

DATA_ROOT = LOCAL_DATA_ROOT
IMAGE_DIR = DATA_ROOT / "images"
TRAIN_XLSX = DATA_ROOT / "partition" / "Test" / "Train.xlsx"
TEST_XLSX = DATA_ROOT / "partition" / "Test" / "Test.xlsx"

categories = ["NC", "G3", "G4", "G5"]

path_images = str(IMAGE_DIR) + "/"
dataframe_train = str(TRAIN_XLSX)
dataframe_test = str(TEST_XLSX)

assert IMAGE_DIR.exists(), f"Image folder not found: {IMAGE_DIR}"
assert TRAIN_XLSX.exists(), f"Train file not found: {TRAIN_XLSX}"
assert TEST_XLSX.exists(), f"Test file not found: {TEST_XLSX}"

print("Dataset ready at:", DATA_ROOT)
print("Number of training entries:", len(pd.read_excel(TRAIN_XLSX)))
print("Number of test entries:", len(pd.read_excel(TEST_XLSX)))

SICAPv2 already exists in local_data.
Dataset ready at: /home/omchakrabarty/local_data/datasets/SICAPv2
Number of training entries: 9959
Number of test entries: 2122


#### **VLM model wrapper**

In [3]:
# Load model and pre-processing tools from HuggingFace.
from transformers import AutoProcessor, AutoModel

# For PLIP, the HuggingFace model ID is "vinid/plip".
processor = AutoProcessor.from_pretrained("vinid/plip")
processor.image_processor.do_center_crop = False

plip = AutoModel.from_pretrained("vinid/plip").eval().to(device)
# We set model in eval mode to avoid dropout inference and batchnorm stats update.

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [4]:
# Again, we will use our PLIP Wrapper for easy interaction.
class PLIPWrapper(torch.nn.Module):
    def __init__(self, encoder, proj_layer):
        super().__init__()
        self.encoder = encoder
        self.proj_layer = proj_layer

    def forward(self, inputs):
        # Forward input through the encoder.
        features = self.encoder(**inputs).pooler_output

        # Project features into the shared image-text embedding space.
        projected_features = self.proj_layer(features)

        # Ensure features are l2-normalized.
        projected_features = projected_features / projected_features.norm(dim=-1, keepdim=True)

        return projected_features

# Create model wrappers for vision and text encoders.
vision_encoder = PLIPWrapper(plip.vision_model, plip.visual_projection).to(device).eval()
text_encoder = PLIPWrapper(plip.text_model, plip.text_projection).to(device).eval()

#### **Compute text prototypes**
(We will need them latter for classification head initialization)

In [5]:
# Ensemble of templates.
templates = ["a histopathology slide showing [CLS]", "histopathology image of [CLS]",
             "pathology tissue showing [CLS]", "presence of [CLS] tissue on image"]

# Category-wise descriptions. These are more informative than class names.
prompts_dict = {"NC": ["benign glands"],
                "G3": ["atrophic dense glands"],
                "G4": ["cribriform ill-formed fused papillary patterns"],
                "G5": ["isolated nest cells without lumen roseting patterns"]}

# Combine all paired options of templates and descriptions.
prompts = {}
for iCategory in categories:
    prompts[iCategory] = [caption.replace("[CLS]", iDescription) for iDescription in prompts_dict[iCategory]
                          for caption in templates]

# Compute embeddings per category.
class_prototypes = []
for iKey in range(len(categories)):
    with torch.no_grad():
        descriptions = prompts[categories[iKey]]
        inputs = processor.tokenizer(
            descriptions,
            max_length=77,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        text_features_ensemble = text_encoder(inputs)

        # Class prototype = average text embedding over prompt ensemble.
        avg_text_features = text_features_ensemble.mean(0).unsqueeze(0)
        avg_text_features = avg_text_features / avg_text_features.norm(dim=-1, keepdim=True)
        class_prototypes.append(avg_text_features)

# Concatenate all class prototypes.
zero_shot_prot = torch.cat(class_prototypes, dim=0).to(device)
print("Zero-shot prototype shape:", zero_shot_prot.shape)

Zero-shot prototype shape: torch.Size([4, 512])


#### **Datasets: few-shot training and test**
Now we are going to modify the vision backbone. Thus, **we cannot pre-compute the deep features, since these will change during adaptation**. Thus, we define the data loader, but we do not pre-compute the features.

In [6]:
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from tqdm.auto import tqdm

# Pre-processing transforms to apply during data loading.
# Keep Resize before ToTensor/Normalize.
plip_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=processor.image_processor.image_mean,
        std=processor.image_processor.image_std
    ),
])

class SICAPv2Dataset(Dataset):
    """Minimal SICAPv2 image-level classification dataset."""
    def __init__(self, dataframe_path, path_images, categories, transforms=None):
        self.df = pd.read_excel(dataframe_path).reset_index(drop=True)
        self.path_images = Path(path_images)
        self.categories = categories
        self.transforms = transforms

        # Convert one-hot columns [NC, G3, G4, G5] to a single class index.
        self.labels = self.df[self.categories].values.argmax(axis=1).astype(np.int64)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_name = self.df.loc[idx, "image_name"]
        image_path = self.path_images / image_name
        image = Image.open(image_path).convert("RGB")

        if self.transforms is not None:
            image = self.transforms(image)

        label = int(self.labels[idx])
        return image, label

def few_shot_subset(dataset, shots=16, seed=1):
    """Select exactly `shots` examples per class from a dataset."""
    rng = np.random.default_rng(seed)
    labels = np.asarray(dataset.labels)
    selected_indices = []

    for class_id in range(len(dataset.categories)):
        class_indices = np.where(labels == class_id)[0]
        rng.shuffle(class_indices)
        selected_indices.extend(class_indices[:shots].tolist())

    rng.shuffle(selected_indices)
    return Subset(dataset, selected_indices)

def train_ft(loader, model, optimizer, epochs=5):
    """Fine-tune selected parameters using mini-batches."""
    model.train()
    loss_history = []

    for epoch in range(epochs):
        epoch_losses = []

        for images, labels in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            images = images.to(device)
            labels = labels.to(device)

            inputs = {"pixel_values": images}

            optimizer.zero_grad(set_to_none=True)
            logits = model(inputs)
            loss = torch.nn.functional.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        avg_loss = float(np.mean(epoch_losses))
        loss_history.append(avg_loss)
        print(f"Epoch {epoch+1:02d}/{epochs} | loss={avg_loss:.4f}")

    return loss_history

@torch.no_grad()
def predict(loader, model):
    """Run mini-batch inference and return probabilities and labels."""
    model.eval()
    all_probs, all_labels = [], []

    for images, labels in tqdm(loader, desc="Predicting", leave=False):
        images = images.to(device)
        inputs = {"pixel_values": images}

        logits = model(inputs)
        probs = torch.softmax(logits, dim=-1)

        all_probs.append(probs.detach().cpu())
        all_labels.append(labels.detach().cpu())

    prob = torch.cat(all_probs, dim=0).numpy()
    y = torch.cat(all_labels, dim=0).numpy()
    return prob, y

def evaluate(y_true, prob):
    """Balanced accuracy and confusion matrix."""
    y_pred = prob.argmax(axis=1)
    aca = balanced_accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(categories))))
    return aca, cm

# Create test loader.
test_dataset = SICAPv2Dataset(
    dataframe_path=dataframe_test,
    path_images=path_images,
    categories=categories,
    transforms=plip_transforms
)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0)

# Create few-shot train loader.
shots, seed = 16, 1
train_full_dataset = SICAPv2Dataset(
    dataframe_path=dataframe_train,
    path_images=path_images,
    categories=categories,
    transforms=plip_transforms
)
train_subset = few_shot_subset(train_full_dataset, shots=shots, seed=seed)
train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, num_workers=0)

print(f"Few-shot training samples: {len(train_subset)} ({shots} shots/class)")
print(f"Test samples: {len(test_dataset)}")

Few-shot training samples: 64 (16 shots/class)
Test samples: 2122


#### **Preliminaries: coupling base model with classification head**
First, we have to equip the vision backbone with a classification head. The best option is to re-use the Linear Probe head explored in the previous notebook.

In [7]:
# We define the classification head.
class ZSLinearProbe(torch.nn.Module):
    def __init__(self, zero_shot_prot, logit_scale):
        super().__init__()
        self.logit_scale = logit_scale
        self.logit_scale.requires_grad = False
        self.prototypes = torch.nn.Parameter(zero_shot_prot.clone())
        # move to device.
        self.to(device)

    def forward(self, features):

        # Get trained prototype
        prototypes = self.prototypes.to(device)

        # l2-normalized trained weights.
        prototypes_norm = prototypes / prototypes.norm(dim=-1, keepdim=True)

        # temparature-scaled similarity per class.
        logit_scale = self.logit_scale.exp()
        logits = features @ prototypes_norm.t() * logit_scale

        return logits
    
    def loss(self, logits, y):
        loss = torch.nn.functional.cross_entropy(logits, y)
        return loss

In [8]:
# Instantiate the classification head, initialized with zero-shot text prototypes.
head = ZSLinearProbe(
    zero_shot_prot=zero_shot_prot.to(device),
    logit_scale=plip.logit_scale.detach().clone().to(device)
)

# Create model combining backbone and classification head.
# This full model is used only for the instructor/optional PEFT demonstration.
model = torch.nn.Sequential(
    copy.deepcopy(vision_encoder),
    head
).to(device).to(torch.float32)

print(model)

Sequential(
  (0): PLIPWrapper(
    (encoder): CLIPVisionModel(
      (embeddings): CLIPVisionEmbeddings(
        (patch_embedding): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
        (position_embedding): Embedding(50, 768)
      )
      (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-11): 12 x CLIPEncoderLayer(
            (self_attn): CLIPAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGELUActivation()
              (fc1): Linear(in_features=7

#### **Preliminaries: counting and freezing parameters**
We present functions to control which parameters are trainable in the network, and which are frozen.

In [9]:
# Auxiliary function to count parameters.
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Auxiliary function to print the number of parameters.
def print_parameters(model):
    print("Number of trainable parameters: " + str(count_parameters(model)))
    for name, param in model.named_parameters():
        if param.requires_grad is True:
            print(name + " " * (70 - len(name)) + " -> Trained:" + str(param.requires_grad))

Lets count the number of trainable parameters currently in the model

In [10]:
print_parameters(model)

Number of trainable parameters: 87851264
0.encoder.embeddings.class_embedding                                   -> Trained:True
0.encoder.embeddings.patch_embedding.weight                            -> Trained:True
0.encoder.embeddings.position_embedding.weight                         -> Trained:True
0.encoder.pre_layrnorm.weight                                          -> Trained:True
0.encoder.pre_layrnorm.bias                                            -> Trained:True
0.encoder.encoder.layers.0.self_attn.k_proj.weight                     -> Trained:True
0.encoder.encoder.layers.0.self_attn.k_proj.bias                       -> Trained:True
0.encoder.encoder.layers.0.self_attn.v_proj.weight                     -> Trained:True
0.encoder.encoder.layers.0.self_attn.v_proj.bias                       -> Trained:True
0.encoder.encoder.layers.0.self_attn.q_proj.weight                     -> Trained:True
0.encoder.encoder.layers.0.self_attn.q_proj.bias                       -> Trained:True
0.

**87.8M parameters!!** And this is an "small" architecture compared with state-of-the-art models. Do we really need to finetune the whole model?

I would say no! In this notebook we will learn how to avoid this challenge with PEFT :)

**First, we will freeze all parameters in the backbone, but the classification head**.

In [11]:
# Freeze all parameters in backbone.
for name, param in model.named_parameters():
    param.requires_grad = False # Freeze.
# Unfreeze classification head.
for name, param in model[1].named_parameters():
    param.requires_grad = True # Unfreeze.
# Print trainable parameters
print_parameters(model)

Number of trainable parameters: 2048
1.prototypes                                                           -> Trained:True


Now, **we will explore different alternatives to selectively or additively adapt the model.**

------
#### **1.5.1. Selective PEFT**

These methods **tune only a small subset of the network**. 
- **Advantadges**: they usually not have specific hyper-parameters, and keep inference times.
- **Drawbacks**: they might distort the pre-trained representations more severely.


#### **Affine-LN Tuning**

**Tuning the Affine parameters from batch [1] or layer [2] normalization layers**, i.e. $\gamma$ and $\beta$. The intuiton behind the methos is: these parameters perform an scaling of such features more relevant for the task at hand, and decrease the scale of features unrelated to the downstream task.

$$\text{Affine} \rightarrow  out=\frac{x-E[x]}{\sqrt{Var[x]+\epsilon}}*\gamma +\beta$$

In [12]:
# Create a copy of the model to be adapted.
model_peft = copy.deepcopy(model).eval().to(device)

# Freeze all parameters first.
for param in model_peft.parameters():
    param.requires_grad = False

# Keep classification head trainable.
for param in model_peft[1].parameters():
    param.requires_grad = True

In [13]:
print("Unfreeze encoder layer norm affine parameters... ", end="\n")

for name, param in model_peft[0].named_parameters():
    if "layer_norm" in name or "layernorm" in name or "layrnorm" in name:
        param.requires_grad = True

# Print trainable parameters.
print_parameters(model_peft)

Unfreeze encoder layer norm affine parameters... 
Number of trainable parameters: 41984
0.encoder.pre_layrnorm.weight                                          -> Trained:True
0.encoder.pre_layrnorm.bias                                            -> Trained:True
0.encoder.encoder.layers.0.layer_norm1.weight                          -> Trained:True
0.encoder.encoder.layers.0.layer_norm1.bias                            -> Trained:True
0.encoder.encoder.layers.0.layer_norm2.weight                          -> Trained:True
0.encoder.encoder.layers.0.layer_norm2.bias                            -> Trained:True
0.encoder.encoder.layers.1.layer_norm1.weight                          -> Trained:True
0.encoder.encoder.layers.1.layer_norm1.bias                            -> Trained:True
0.encoder.encoder.layers.1.layer_norm2.weight                          -> Trained:True
0.encoder.encoder.layers.1.layer_norm2.bias                            -> Trained:True
0.encoder.encoder.layers.2.layer_norm1.wei

In [14]:
# Train the selected parameters of the model.
# For a fast live demo, keep epochs small. Increase to 20 for a longer run.
epochs, batch_size, learning_rate = 5, 16, 0.001

optimizer = torch.optim.Adam(
    [p for p in model_peft.parameters() if p.requires_grad],
    lr=learning_rate
)

train_ft(loader=train_loader, model=model_peft, optimizer=optimizer, epochs=epochs)

Epoch 1/5:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 01/5 | loss=1.5720


Epoch 2/5:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 02/5 | loss=1.0403


Epoch 3/5:   0%|          | 0/4 [00:00<?, ?it/s]

...

Epoch 4/5:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 04/5 | loss=0.6090


Epoch 5/5:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 05/5 | loss=0.5486


[1.5720035433769226,
 1.0403276234865189,
 0.8617651462554932,
 0.6090172827243805,
 0.5486192181706429]

In [15]:
# Predict on the test set using mini-batches.
prob, Y_test = predict(test_loader, model_peft)

Predicting:   0%|          | 0/266 [00:00<?, ?it/s]

In [16]:
# Compute metrics.
aca, cm = evaluate(Y_test, prob)
print("Balanced accuracy: " + str(aca))
print("Confusion matrix: ")
print(str(cm))

Balanced accuracy: 0.5942205688750751
Confusion matrix: 
[[324  19 211  90]
 [ 26 190 106  71]
 [ 14 107 355 377]
 [  0   0   6 226]]


#### **Bias Tuning**

**Tuning only the Bias parameters** in ViTs, i.e. BitFit [3] has shown promising performance compared to full fine-tuning.


In [17]:
# Create a copy of the model to be adapted.
model_peft = copy.deepcopy(model).eval().to(device)

# Freeze all parameters first.
for param in model_peft.parameters():
    param.requires_grad = False

# Keep classification head trainable.
for param in model_peft[1].parameters():
    param.requires_grad = True

In [18]:
print("Unfreeze bias parameters... ", end="\n")

for name, param in model_peft[0].named_parameters():
    if "bias" in name:
        param.requires_grad = True

# Print trainable parameters.
print_parameters(model_peft)

Unfreeze bias parameters... 
Number of trainable parameters: 104960
0.encoder.pre_layrnorm.bias                                            -> Trained:True
0.encoder.encoder.layers.0.self_attn.k_proj.bias                       -> Trained:True
0.encoder.encoder.layers.0.self_attn.v_proj.bias                       -> Trained:True
0.encoder.encoder.layers.0.self_attn.q_proj.bias                       -> Trained:True
0.encoder.encoder.layers.0.self_attn.out_proj.bias                     -> Trained:True
0.encoder.encoder.layers.0.layer_norm1.bias                            -> Trained:True
0.encoder.encoder.layers.0.mlp.fc1.bias                                -> Trained:True
0.encoder.encoder.layers.0.mlp.fc2.bias                                -> Trained:True
0.encoder.encoder.layers.0.layer_norm2.bias                            -> Trained:True
0.encoder.encoder.layers.1.self_attn.k_proj.bias                       -> Trained:True
0.encoder.encoder.layers.1.self_attn.v_proj.bias              

In [19]:
# Train the selected parameters of the model.
# For a fast live demo, keep epochs small. Increase to 20 for a longer run.
epochs, batch_size, learning_rate = 5, 16, 0.001

optimizer = torch.optim.Adam(
    [p for p in model_peft.parameters() if p.requires_grad],
    lr=learning_rate
)

train_ft(loader=train_loader, model=model_peft, optimizer=optimizer, epochs=epochs)

Epoch 1/5:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 01/5 | loss=1.5935


Epoch 2/5:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 02/5 | loss=1.1051


Epoch 3/5:   0%|          | 0/4 [00:00<?, ?it/s]

...

Epoch 4/5:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 04/5 | loss=0.4814


Epoch 5/5:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 05/5 | loss=0.3746


[1.593500018119812,
 1.105121225118637,
 0.6459078937768936,
 0.48142044246196747,
 0.37463074922561646]

In [ ]:
# Predict on the test set using mini-batches.
prob, Y_test = predict(test_loader, model_peft)

Predicting:   0%|          | 0/266 [00:00<?, ?it/s]

In [ ]:
# Compute metrics.
aca, cm = evaluate(Y_test, prob)
print("Balanced accuracy: " + str(aca))
print("Confusion matrix: ")
print(str(cm))

------
#### **1.5.2. Additive PEFT**

These methods add an small set of parameters, so-called Adapters. Usually, they perform residual modifications of pre-trained features. Nowadays, the most popular method is LoRA, which performs a low-rank adaptation.
 
- **Advantadges**: they are more flexible than selective methods, since you can control how many parameters you introduce. The residual feature modification produces smoother changes.
- **Drawbacks**: You need to set the number of parameters you introduce, which produce extra hyper-parameters. Some Adapters also increase inference time, since you are adding operations. Note that this is not neccesary the case of LoRA, since it can be computed in paralel.

LoRA introduces to new matrices, A and B, which perform a residual modification on the output of an specific weight. Given a base linear weight $W$, and an input feature representation $x$, we can formalize LoRA as:

$$out = W(x) + B(A(x))$$


Where A and B are low-rank matrices.

**How parameter-efficient are Low-rank Adapters?** Let us denote that $x$ has dimensionality of $D$ features, and $W$ is a Linear layer with $D$ features. If we were to use only one full-rank layer for the residual modification, which we denote as $W'$, such that $out = W(x) + W'(x)$, this new later would introduce $D\times D$ parameters. Instead, the low-rank matrices A and B, with rank $r$ (e.g. $r=4$), have the dimensionality $A(D\times r)$, and $B(r \times D)$, such that A firstly compress the embedding in $r$ features, and B return it to the original dimensionality. Thus, the number of introduced parameters are $2 \cdot r \cdot D$. Image $D=128$, and typically, $r=4$. A basic Adapter would introduce 16.3K parameters, while LoRA introduces 1K.

**Numbers apart, let's see how it works!**


In [ ]:
# Create the LoRA layer, to replace any linear weight
class _LoRALayer(torch.nn.Module):
    def __init__(self, w, w_a, w_b):
        super().__init__()
        self.w = w      # Original weight.
        self.w_a = w_a  # Matrix A.
        self.w_b = w_b  # Matrix B.

    def forward(self, x):
        x = self.w(x) + self.w_b(self.w_a(x)) # Residual modification with Adapter.
        return x

In [ ]:
# Create LoRA Wrapper
class LoRAWrapper(torch.nn.Module):
    def __init__(self, vit_model, r=4):
        super(LoRAWrapper, self).__init__()
        # Inits
        self.ViTbase = vit_model # ViT to modify
        self.r = r               # Rank
        # create for storage, then we can init them or load weights.
        self.w_As = []  # Storage for linear layers of A matrices.
        self.w_Bs = []  # Storage for linear layers of B matrices.
        
        # We go trough the base encoder, detect Multi-Head Attention blocks, and modify adding the Adapters.
        for i, layer in enumerate(list(list(self.ViTbase.encoder.children())[2].modules())):
            if layer._get_name() == 'CLIPAttention':  # Multi-Head Attention Blocks.

                # k_proj (key)
                w_a_linear_qkv = torch.nn.Linear(layer.k_proj.in_features, r, bias=False) # layer for A matrix.
                w_b_linear_qkv = torch.nn.Linear(r, layer.k_proj.in_features, bias=False) # layer for B matrix.
                torch.nn.init.zeros_(w_b_linear_qkv.weight)                               # Set values in B to 0s.
                self.w_As.append(w_a_linear_qkv), self.w_Bs.append(w_b_linear_qkv)        # Store new weights.
                layer.k_proj = _LoRALayer(layer.k_proj, w_a_linear_qkv, w_b_linear_qkv)   # Modify layer with LoRA layer.

                # v_proj (query)
                w_a_linear_qkv = torch.nn.Linear(layer.v_proj.in_features, r, bias=False) # layer for A matrix.
                w_b_linear_qkv = torch.nn.Linear(r, layer.v_proj.in_features, bias=False) # layer for B matrix.
                torch.nn.init.zeros_(w_b_linear_qkv.weight)                               # Set values in B to 0s.
                self.w_As.append(w_a_linear_qkv), self.w_Bs.append(w_b_linear_qkv)        # Store new weights.
                layer.v_proj = _LoRALayer(layer.v_proj, w_a_linear_qkv, w_b_linear_qkv)   # Modify layer with LoRA layer.

                # q_proj (value)
                w_a_linear_qkv = torch.nn.Linear(layer.q_proj.in_features, r, bias=False) # layer for A matrix.
                w_b_linear_qkv = torch.nn.Linear(r, layer.q_proj.in_features, bias=False) # layer for B matrix.
                torch.nn.init.zeros_(w_b_linear_qkv.weight)                               # Set values in B to 0s.
                self.w_As.append(w_a_linear_qkv), self.w_Bs.append(w_b_linear_qkv)        # Store new weights.
                layer.q_proj = _LoRALayer(layer.q_proj, w_a_linear_qkv, w_b_linear_qkv)   # Modify layer with LoRA layer.

    def forward(self, x):
        return self.ViTbase(x)

In [ ]:
# Create a copy of the model to be adapted.
model_peft = copy.deepcopy(model).eval().to(device)

# Freeze all existing parameters first.
for param in model_peft.parameters():
    param.requires_grad = False

# Add LoRA Wrapper to the vision backbone.
r = 4
model_peft[0] = LoRAWrapper(model_peft[0], r=r).to(device)

# Keep classification head trainable.
for param in model_peft[1].parameters():
    param.requires_grad = True

# LoRA A/B matrices are newly created and should be trainable.
for module in model_peft[0].modules():
    if isinstance(module, _LoRALayer):
        for param in module.w_a.parameters():
            param.requires_grad = True
        for param in module.w_b.parameters():
            param.requires_grad = True

# Print trainable parameters.
print_parameters(model_peft)

In [ ]:
# Train the selected parameters of the model.
# For a fast live demo, keep epochs small. Increase to 20 for a longer run.
epochs, batch_size, learning_rate = 5, 16, 0.001

optimizer = torch.optim.Adam(
    [p for p in model_peft.parameters() if p.requires_grad],
    lr=learning_rate
)

train_ft(loader=train_loader, model=model_peft, optimizer=optimizer, epochs=epochs)

In [ ]:
# Predict on the test set using mini-batches.
prob, Y_test = predict(test_loader, model_peft)

In [ ]:
# Compute metrics.
aca, cm = evaluate(Y_test, prob)
print("Balanced accuracy: " + str(aca))
print("Confusion matrix: ")
print(str(cm))

--- 
##### **ACTIVITY**

Well, now you know the basics of PEFT methods. If you want to know more, I reccomend:

- How is the performance if you do not initialize the B matrix with 0s in LoRA? How rank modification in LoRA affects the performance?
- Doing early stopping based on validation data also helps on avoiding over-fitting during PEFT. Modify the loaders function to create a few-shot dataset for validation, and modify training to save the best model based on validation loss.
- Explore the comparison with Black-box Adapters for more and less than 16 shots.
- Try developing the same pipeline for [CONCH](https://huggingface.co/MahmoodLab/CONCH) [4], a revently introduced VLM for histology. Its vision backbone is large scale (ViT-B/16), takes large-resolution input images (448x448), and is pre-trained with more data. How does model scaling translates to PEFT? As you can see, the larger the network, the more convinient is PEFT with respet to full fine-tuning.


--- 
## **References**


[1] Frankle, J., Schwab, D. J., Morcos, A. S. (2021). Training batchnorm and only batchnorm: On the expressive power of random features in cnns. International Conference on Learning Representations (ICLR). \
[2] Ben-Zaken, E., Ravfogel, S., Goldberg, Y. (2021). Bitfit: Simple parameter efficient fine-tuning for transformer-based masked language-models. Association for Computational Linguistics. \
[3] Hu, E. J., et al., (2022). LoRA: Low-rank adaptation of large language models. International Conference on Learning Representations (ICLR). \
[4] Lu, M.Y., Chen, B., Williamson, D.F.K. et al. (2024) A visual-language foundation model for computational pathology. Nature Medicine.

--- 